# 05 - Visualización Interactiva con Plotly
## Sistema de Predicción Temprana de Plagas - Sierra del Patlachique

**Objetivo:** Crear visualizaciones interactivas para explorar indicadores de riesgo

**Responsabilidades:**
- ✓ Gráficas interactivas de series temporales
- ✓ Visualizar distribuciones de variables
- ✓ Gráficas de correlaciones
- ✓ Mapa de calor de indicadores
- ✓ Análisis por tipo de plaga
- ✓ Dashboard resumen

**NO hacer aquí:**
- ✗ Modelado predictivo
- ✗ Entrenamiento de ML

**Entrada:** `data/features/datos_patlachique_features.csv`

**Salida:** Gráficas interactivas en HTML (visualización exploratoria)

In [3]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("✓ Librerías cargadas correctamente")

# Cargar datos con features
df = pd.read_csv("../data/features/datos_patlachique_features.csv",
                  index_col='fecha', parse_dates=True)

print(f"✓ Datos cargados: {df.shape}")
print(f"  - Registros: {len(df)}")
print(f"  - Variables: {df.shape[1]}")
print(f"  - Período: {df.index.min().date()} a {df.index.max().date()}")

✓ Librerías cargadas correctamente
✓ Datos cargados: (817, 21)
  - Registros: 817
  - Variables: 21
  - Período: 2024-01-01 a 2026-03-27


## 1. GRÁFICA DE SERIE TEMPORAL - TEMPERATURA

**Visualización:** Línea interactiva con área sombreada

**Qué muestra:**
- Variación diaria de temperatura
- Promedio móvil de 7 días (tendencia)
- Rango mínimo-máximo histórico

In [4]:
print("\n" + "="*70)
print("1. SERIE TEMPORAL - TEMPERATURA")
print("="*70)

fig = go.Figure()

# Temperatura diaria
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['temp_media_C'],
    mode='lines',
    name='Temperatura Diaria',
    line=dict(color='#FF6B6B', width=1),
    opacity=0.6
))

# Temperatura promedio 7 días
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['temp_media_7d'],
    mode='lines',
    name='Promedio 7 días',
    line=dict(color='#FF4444', width=2),
    opacity=0.9
))

# Rango histórico (mínimo y máximo)
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['temp_media_C'].rolling(30).max(),
    mode='lines',
    name='Máximo (30d)',
    line=dict(color='rgba(255, 100, 100, 0)'),
    showlegend=False
))

fig.add_trace(go.Scatter(
    x=df.index,
    y=df['temp_media_C'].rolling(30).min(),
    mode='lines',
    name='Mínimo (30d)',
    line=dict(color='rgba(255, 100, 100, 0)'),
    fillcolor='rgba(255, 100, 100, 0.2)',
    fill='tonexty'
))

fig.update_layout(
    title='Temperatura Media a lo Largo del Tiempo',
    xaxis_title='Fecha',
    yaxis_title='Temperatura (°C)',
    hovermode='x unified',
    height=500,
    template='plotly_white'
)

fig.show()
fig.write_html("../data/reports/01_temperatura_temporal.html")
print("✓ Gráfica guardada: reports/01_temperatura_temporal.html")


1. SERIE TEMPORAL - TEMPERATURA


✓ Gráfica guardada: reports/01_temperatura_temporal.html


## 2. SERIE TEMPORAL - LLUVIA

**Visualización:** Barras + línea de acumulado

**Qué muestra:**
- Lluvia diaria (barras)
- Lluvia acumulada en 7 días (línea)
- Identifica períodos secos vs lluviosos

In [5]:
print("\n" + "="*70)
print("2. SERIE TEMPORAL - LLUVIA")
print("="*70)

# Crear figura con eje secundario
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Lluvia diaria (barras)
fig.add_trace(
    go.Bar(
        x=df.index,
        y=df['lluvia_mm'],
        name='Lluvia Diaria',
        marker_color='#4A90E2',
        opacity=0.6
    ),
    secondary_y=False,
)

# Lluvia acumulada 7 días (línea)
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df['lluvia_7d'],
        mode='lines',
        name='Acumulado 7d',
        line=dict(color='#2E5C8A', width=2)
    ),
    secondary_y=True,
)

fig.update_layout(
    title='Lluvia: Diaria vs Acumulada',
    xaxis_title='Fecha',
    hovermode='x unified',
    height=500,
    template='plotly_white'
)

fig.update_yaxes(title_text="Lluvia Diaria (mm)", secondary_y=False)
fig.update_yaxes(title_text="Lluvia Acumulada 7d (mm)", secondary_y=True)

fig.show()
fig.write_html("../data/reports/02_lluvia_temporal.html")
print("✓ Gráfica guardada: reports/02_lluvia_temporal.html")


2. SERIE TEMPORAL - LLUVIA


✓ Gráfica guardada: reports/02_lluvia_temporal.html


## 3. SERIE TEMPORAL - HUMEDAD RELATIVA

**Visualización:** Línea con banda de riesgo

**Qué muestra:**
- Humedad relativa diaria
- Zona de riesgo (HR < 70%)
- Períodos de mayor peligro para pulgones y ácaros

In [6]:
print("\n" + "="*70)
print("3. SERIE TEMPORAL - HUMEDAD RELATIVA")
print("="*70)

fig = go.Figure()

# Humedad relativa
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['humedad_relativa_pct'],
    mode='lines',
    name='Humedad Relativa',
    line=dict(color='#2ECC71', width=2),
    fill='tozeroy',
    fillcolor='rgba(46, 204, 113, 0.1)'
))

# Línea de umbral crítico (70%)
fig.add_hline(
    y=70,
    line_dash="dash",
    line_color="red",
    annotation_text="Umbral crítico (70%)",
    annotation_position="right"
)

# Línea de umbral muy crítico (60%)
fig.add_hline(
    y=60,
    line_dash="dot",
    line_color="darkred",
    annotation_text="Muy crítico (60%)",
    annotation_position="right"
)

fig.update_layout(
    title='Humedad Relativa - Zonas de Riesgo',
    xaxis_title='Fecha',
    yaxis_title='Humedad Relativa (%)',
    hovermode='x unified',
    height=500,
    template='plotly_white',
    shapes=[
        dict(
            type="rect",
            xref="paper",
            yref="y",
            x0=0, x1=1,
            y0=0, y1=60,
            fillcolor="red",
            opacity=0.1,
            layer="below",
            line_width=0,
        )
    ]
)

fig.show()
fig.write_html("../data/reports/03_humedad_relativa_temporal.html")
print("✓ Gráfica guardada: reports/03_humedad_relativa_temporal.html")


3. SERIE TEMPORAL - HUMEDAD RELATIVA


✓ Gráfica guardada: reports/03_humedad_relativa_temporal.html


## 4. INDICADORES DE RIESGO POR PLAGA

**Visualización:** Área apilada mostrando todas las plagas

**Qué muestra:**
- Cuándo cada plaga está en riesgo
- Períodos de múltiples plagas simultáneas
- Tendencias a lo largo del año

In [7]:
print("\n" + "="*70)
print("4. INDICADORES DE RIESGO POR PLAGA")
print("="*70)

fig = go.Figure()

# Cada plaga es un área
plagas = {
    'riesgo_chapulin': 'Chapulín',
    'riesgo_pulgon': 'Pulgón',
    'riesgo_cogollero': 'Cogollero',
    'riesgo_arana_roja': 'Araña Roja'
}

colores = {
    'riesgo_chapulin': '#E74C3C',
    'riesgo_pulgon': '#F39C12',
    'riesgo_cogollero': '#9B59B6',
    'riesgo_arana_roja': '#E67E22'
}

for col, nombre in plagas.items():
    fig.add_trace(go.Scatter(
        x=df.index,
        y=df[col],
        mode='lines',
        name=nombre,
        line=dict(width=2, color=colores[col]),
        stackgroup='riesgo'
    ))

fig.update_layout(
    title='Indicadores de Riesgo por Plaga',
    xaxis_title='Fecha',
    yaxis_title='Presencia de Riesgo (0=No, 1=Sí)',
    hovermode='x unified',
    height=500,
    template='plotly_white'
)

fig.show()
fig.write_html("../data/reports/04_riesgos_plagas.html")
print("✓ Gráfica guardada: reports/04_riesgos_plagas.html")


4. INDICADORES DE RIESGO POR PLAGA


✓ Gráfica guardada: reports/04_riesgos_plagas.html


## 5. NIVEL DE ALERTA - MAPA DE CALOR

**Visualización:** Calendario con colores de riesgo

**Qué muestra:**
- Cada día coloreado según nivel de alerta
- Verde = sin riesgo
- Rojo = riesgo crítico
- Ideal para ver períodos de mayor peligro

In [8]:
print("\n" + "="*70)
print("5. MAPA DE CALOR - NIVEL DE ALERTA")
print("="*70)

# Crear matriz mes-día
df_temp = df.copy()
df_temp['mes'] = df_temp.index.month
df_temp['día'] = df_temp.index.day
df_temp['semana'] = df_temp.index.isocalendar().week

# Mapa de calor
fig = px.density_heatmap(
    df_temp,
    x='mes',
    y='nivel_alerta',
    nbinsx=12,
    nbinsy=4,
    color_continuous_scale='RdYlGn_r',
    title='Mapa de Calor: Nivel de Alerta por Mes',
    labels={'mes': 'Mes', 'nivel_alerta': 'Nivel de Alerta'}
)

fig.update_layout(
    height=400,
    template='plotly_white',
    xaxis=dict(
        tickvals=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
        ticktext=['E', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D']
    )
)

fig.show()
fig.write_html("../data/reports/05_mapa_calor_alerta.html")
print("✓ Gráfica guardada: reports/05_mapa_calor_alerta.html")


5. MAPA DE CALOR - NIVEL DE ALERTA


✓ Gráfica guardada: reports/05_mapa_calor_alerta.html


## 6. DISTRIBUCIONES - HISTOGRAMAS

**Visualización:** Histogramas interactivos

**Qué muestra:**
- Distribución de cada variable
- Valores más frecuentes
- Sesgo de los datos

In [9]:
print("\n" + "="*70)
print("6. DISTRIBUCIONES - HISTOGRAMAS")
print("="*70)

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Temperatura (°C)', 'Lluvia (mm)', 'Humedad Suelo', 'Humedad Relativa (%)')
)

# Temperatura
fig.add_trace(
    go.Histogram(x=df['temp_media_C'], name='Temperatura', nbinsx=30, marker_color='#FF6B6B'),
    row=1, col=1
)

# Lluvia
fig.add_trace(
    go.Histogram(x=df['lluvia_mm'], name='Lluvia', nbinsx=30, marker_color='#4A90E2'),
    row=1, col=2
)

# Humedad suelo
fig.add_trace(
    go.Histogram(x=df['humedad_suelo_frac'], name='Humedad Suelo', nbinsx=30, marker_color='#95A5A6'),
    row=2, col=1
)

# Humedad relativa
fig.add_trace(
    go.Histogram(x=df['humedad_relativa_pct'], name='Humedad Relativa', nbinsx=30, marker_color='#2ECC71'),
    row=2, col=2
)

fig.update_layout(height=600, showlegend=False, title_text="Distribuciones de Variables", template='plotly_white')
fig.update_xaxes(title_text="Temperatura (°C)", row=1, col=1)
fig.update_xaxes(title_text="Lluvia (mm)", row=1, col=2)
fig.update_xaxes(title_text="Humedad Suelo", row=2, col=1)
fig.update_xaxes(title_text="Humedad Relativa (%)", row=2, col=2)

fig.show()
fig.write_html("../data/reports/06_distribuciones.html")
print("✓ Gráfica guardada: reports/06_distribuciones.html")


6. DISTRIBUCIONES - HISTOGRAMAS


✓ Gráfica guardada: reports/06_distribuciones.html


## 7. CORRELACIONES - MATRIZ INTERACTIVA

**Visualización:** Heatmap de correlaciones

**Qué muestra:**
- Relaciones entre todas las variables
- Positivas (azul) y negativas (rojo)
- Identifica variables clave

In [10]:
print("\n" + "="*70)
print("7. MATRIZ DE CORRELACIONES")
print("="*70)

# Seleccionar variables numéricas
cols_for_corr = ['temp_media_C', 'lluvia_mm', 'humedad_suelo_frac', 'humedad_relativa_pct',
                 'lluvia_7d', 'lluvia_21d', 'dias_secos_14d', 'temp_media_7d', 'hr_media_7d']

corr_matrix = df[cols_for_corr].corr()

fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    colorscale='RdBu',
    zmid=0,
    text=np.round(corr_matrix.values, 2),
    texttemplate='%{text}',
    textfont={"size": 10},
    colorbar=dict(title="Correlación")
))

fig.update_layout(
    title='Matriz de Correlaciones de Variables',
    xaxis_title='Variables',
    yaxis_title='Variables',
    height=600,
    width=700
)

fig.show()
fig.write_html("../data/reports/07_correlaciones.html")
print("✓ Gráfica guardada: reports/07_correlaciones.html")


7. MATRIZ DE CORRELACIONES


✓ Gráfica guardada: reports/07_correlaciones.html


## 8. RESUMEN ESTADÍSTICO

**Visualización:** Tabla interactiva con estadísticas

**Qué muestra:**
- Media, desviación estándar, mínimo, máximo
- Percentiles importantes
- Resumen completo de cada variable

In [11]:
print("\n" + "="*70)
print("8. RESUMEN ESTADÍSTICO")
print("="*70)

# Estadísticas descriptivas
stats_df = df[['temp_media_C', 'lluvia_mm', 'humedad_suelo_frac', 'humedad_relativa_pct']].describe().T
stats_df = stats_df.round(2)

# Crear tabla
fig = go.Figure(data=[go.Table(
    header=dict(
        values=['Variable'] + list(stats_df.columns),
        fill_color='paleturquoise',
        align='left',
        font=dict(color='black', size=12)
    ),
    cells=dict(
        values=[stats_df.index] + [stats_df[col] for col in stats_df.columns],
        fill_color='lavender',
        align='left',
        font=dict(color='black', size=11)
    )
)])

fig.update_layout(
    title='Estadísticas Descriptivas de Variables',
    height=400,
    template='plotly_white'
)

fig.show()
fig.write_html("../data/reports/08_estadisticas.html")
print("✓ Gráfica guardada: reports/08_estadisticas.html")

print("\n" + "="*70)
print("✅ VISUALIZACIÓN COMPLETADA")
print("="*70)
print("""
Se crearon 8 gráficas interactivas en HTML:
  ✓ 01_temperatura_temporal.html
  ✓ 02_lluvia_temporal.html
  ✓ 03_humedad_relativa_temporal.html
  ✓ 04_riesgos_plagas.html
  ✓ 05_mapa_calor_alerta.html
  ✓ 06_distribuciones.html
  ✓ 07_correlaciones.html
  ✓ 08_estadisticas.html

Abre cualquiera en tu navegador para explorar los datos.

Próximo paso: Notebook 06 - Modelos Predictivos
""")


8. RESUMEN ESTADÍSTICO


✓ Gráfica guardada: reports/08_estadisticas.html

✅ VISUALIZACIÓN COMPLETADA

Se crearon 8 gráficas interactivas en HTML:
  ✓ 01_temperatura_temporal.html
  ✓ 02_lluvia_temporal.html
  ✓ 03_humedad_relativa_temporal.html
  ✓ 04_riesgos_plagas.html
  ✓ 05_mapa_calor_alerta.html
  ✓ 06_distribuciones.html
  ✓ 07_correlaciones.html
  ✓ 08_estadisticas.html

Abre cualquiera en tu navegador para explorar los datos.

Próximo paso: Notebook 06 - Modelos Predictivos

